# Milestone 2: RAG Pipeline Exploration

## 1. Setup and Imports

Set up project paths, import retrieval utilities, and check that required artifacts exist.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "processed_data_sample.parquet"
SEMANTIC_STORE_PATH = PROJECT_ROOT / "data" / "processed" / "faiss_store"
BM25_PATH = PROJECT_ROOT / "data" / "processed" / "bm25_retriever.pkl"

from src.semantic import get_or_build_vectorstore, get_semantic_retriever
from src.bm25 import load_bm25_retriever

assert DATA_PATH.exists(), f"Missing parquet file: {DATA_PATH}"
assert SEMANTIC_STORE_PATH.exists(), f"Missing semantic store: {SEMANTIC_STORE_PATH}"

print("Setup complete.")

/Users/sashas/miniforge3/envs/retrieval-app/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


## 2. Load Retrieval Components 

Load the saved semantic vector store and BM25 retriever.

In [2]:
semantic_vectorstore = get_or_build_vectorstore(
    data_path=str(DATA_PATH),
    store_path=str(SEMANTIC_STORE_PATH),
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

bm25 = load_bm25_retriever(str(BM25_PATH))

print("BM25 retriever loaded." if bm25 is not None else "BM25 retriever not found.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9352.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded semantic vector store.
BM25 retriever loaded.


## 3. Semantic Retrieval Sanity Checks

Create a semantic retriever and inspect a few retrieved documents to confirm that retrieval is working as expected. 

In [3]:
semantic_retriever = get_semantic_retriever(semantic_vectorstore, top_k=5)
print("Semantic retriever ready.")

Semantic retriever ready.


In [4]:
# Test Semantic Retriever
query = "good gift for someone who loves gardening"
docs = semantic_retriever.invoke(query)

print(f"Retrieved {len(docs)} documents for test query.")

Retrieved 5 documents for test query.


In [5]:
test_queries = [
    "blue patio chair cushion",
    "good gift for someone who loves gardening",
    "something to make a garden look more colorful and lively"
]

for q in test_queries:
    retrieved_docs = semantic_retriever.invoke(q)
    print(f"\nQUERY: {q}")
    for i, doc in enumerate(retrieved_docs[:2], start=1):
        print(f"\nResult {i}")
        print("Title:", doc.metadata.get("product_title"))
        print("Review:", doc.metadata.get("review_text"))
        print("Rating:", doc.metadata.get("rating"))


QUERY: blue patio chair cushion

Result 1
Title: Pillow Perfect Stripe Indoor/Outdoor 1 Piece Split Back Round Corner Chair Seat Cushion with Ties, Deep Seat, Weather, and Fade Resistant, 40.5" x 21", Red/White Midland, 1 Count
Review: I must have the oddest-sized patio chairs ever made because I could not find replacement cushions for them. I finally looked on Amazon and found these. They are still just a tad bit too big, but they work and were a decent price, as well. I was also trying to match my existing umbrella. I am really pleased with my purchase.
Rating: 5

Result 2
Title: Rocking Chair Cushion Pad, 47IN Indoor Outdoor Navy Blue Rocking Chair Cushions Set Soft Thickened Patio Chaise Lounger Cushion Overstuffed Patio Chair Cushion
Review: Very flimsy, no comfort at all, purchased navy blue which was very pretty but within one month it faded to grey. Very disappointing.
Rating: 1

QUERY: good gift for someone who loves gardening

Result 1
Title: Funny Combat Garden Gnome Riding

The retrieved results appeared relevant across both keyword-heavy and more semantic queries, suggesting that the FAISS-based semantic retriever was functioning as intended.

## 4. Context Building

Format retrieved documents into a structured context block for the LLM.

In [6]:
def build_context(docs):
    """
    Convert retrieved documents into a structured context block for RAG.
    """
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        block = (
            f"Document {i}\n"
            f"Product title: {doc.metadata.get('product_title', 'N/A')}\n"
            f"Review title: {doc.metadata.get('review_title', 'N/A')}\n"
            f"Review text: {doc.metadata.get('review_text', 'N/A')}\n"
            f"Review evidence:\n"
            f"- Review rating: {doc.metadata.get('rating', 'N/A')}\n"
            f"- Average product rating: {doc.metadata.get('average_rating', 'N/A')}\n"
            f"- Number of ratings: {doc.metadata.get('rating_number', 'N/A')}\n"
            f"- Price: {doc.metadata.get('price', 'N/A')}\n"
            f"- Categories: {doc.metadata.get('categories', 'N/A')}\n"
            f"- Features: {doc.metadata.get('features', 'N/A')}\n"
            f"- Description: {doc.metadata.get('description', 'N/A')[:100]}"
        )
        context_blocks.append(block)

    return "\n\n---\n\n".join(context_blocks)

In [7]:
context = build_context(docs)
print(context[:2000])

Document 1
Product title: Funny Combat Garden Gnome Riding a Flamingo – Cute Home Decor Statue, for Lawn, Yard, or Office - Indoor Outdoor 11 Inch Tall, by Dodo Creations
Review title: Perfect
Review text: It is a nice size
Review evidence:
- Review rating: 5
- Average product rating: 4.0
- Number of ratings: 76
- Price: N/A
- Categories: Patio, Lawn & Garden Outdoor Décor Garden Sculptures & Statues Outdoor Statues
- Features: 💗 ARMED WITH CUTENESS -Atop a military pink flamingo, our gnome is an expert in gardens security. 💗 FUNNY DECOR -Our decorative figures make an eye-catching ornament. Perfect on your desk or shelf. 💗 DURABLE & WEATHERPROOF - Poly-resin statues able to withstand elements and changes in season. 💗 AN AWESOME GIFT - This garden figurine makes a unique gift for family, friends or neighbors 💗 BATTLE-READY PACKAGING - Our gnome is carefully packed and shipped in a safe polyfoam box.
- Description: Struggling to find the perfect gift for an avid gardener? Choosing gifts

Including rating, average rating, and number of ratings in the context made it easier to instruct the model to weigh both qualitative review evidence and quantitative support when making recommendations.

Because of token limit constraints in the hybrid RAG pipeline, the description field was modified by truncating it to the first 100 characters. This helps significantly reduce the overall context size while still preserving a small amount of useful product information.

## 5. LLM Setup

Use a hosted Groq model for fast iterative experimentation without requiring local GPU resources.

In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

print("API key loaded:", os.getenv("GROQ_API_KEY") is not None)

API key loaded: True


In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant")

response = llm.invoke("In one sentence, what makes a product review useful?")

print(response.content)

A product review is considered useful when it provides a balanced and detailed assessment of a product's features, performance, and any potential drawbacks, based on the reviewer's personal experience and expertise.


We selected `llama-3.1-8b-instant` via Groq because it provides a strong balance of response quality and very fast inference, making it well-suited for iterative RAG development on a local machine without a GPU.

## 6. Prompt Template Experiments

Test three prompt styles: a loose baseline, a strict grounded prompt, and a balanced prompt designed to support concise but useful recommendations.

In [10]:
SYSTEM_PROMPT_1 = """
You are a helpful Amazon shopping assistant.

Answer the user's question using the provided product reviews and metadata.
Provide helpful suggestions and explain your reasoning.
"""

SYSTEM_PROMPT_2 = """
You are a helpful Amazon shopping assistant.

Answer the user's question using ONLY the provided Amazon product review and metadata context.
Do not make up any information.

If the context is not sufficient to answer confidently, say that the available reviews do not provide enough information.

Keep the answer short, factual, and grounded in the retrieved documents.
"""

SYSTEM_PROMPT_3 = """
You are a helpful Amazon shopping assistant.

Answer the user's question using ONLY the provided Amazon product review and metadata context.
Do not make up details that are not supported by the context.
If the context is not sufficient to answer confidently, say so.

When answering:
- keep the answer concise and practical
- recommend only products supported by the retrieved context
- mention product titles when helpful
- avoid repeating duplicate or near-duplicate products
- give 2 to 4 recommendations when several good options exist
""" 


In [11]:
def build_prompt(query, context, system_prompt):
    """
    Build the final RAG prompt from the system prompt, retrieved context,
    and user query.
    """
    return f"""{system_prompt}

Context:
{context}

Question:
{query}

Answer:"""

## 7. Semantic RAG Prototype

Build and test a simple semantic RAG function before moving the final implementation into `src/rag_pipeline.py`.


In [12]:
def run_semantic_rag(query, retriever, llm, system_prompt):
    """
    Run a semantic RAG pipeline:
    query -> retrieve documents -> build context -> build prompt -> generate answer
    """
    docs = retriever.invoke(query)
    context = build_context(docs)
    prompt = build_prompt(query, context, system_prompt)
    response = llm.invoke(prompt)

    answer = response.content if hasattr(response, "content") else str(response)

    return {
        "query": query,
        "answer": answer,
        "docs": docs,
        "context": context,
        "prompt": prompt,
    }

In [13]:
query = "What would be a good gift for someone who loves gardening?"

for i, system_prompt in enumerate(
    [SYSTEM_PROMPT_1, SYSTEM_PROMPT_2, SYSTEM_PROMPT_3], start=1
):
    result = run_semantic_rag(
        query=query,
        retriever=semantic_retriever,
        llm=llm,
        system_prompt=system_prompt
    )

    print(f"\nPROMPT VARIANT {i}")
    print(result["answer"])
    print("-" * 80)


PROMPT VARIANT 1
Based on the product reviews and metadata, I would recommend the following options as good gifts for someone who loves gardening:

1. **Garden Tools Set, JUMPHIGH 10 Pieces Gardening Tools with Purple Floral Print** (Document 4): This tool set has an average rating of 4.4 out of 5 stars and has received over 3,200 reviews. It includes a variety of tools that can be used for different gardening tasks, and it's available at an affordable price of $35.99.
2. **Scuddles Garden Tools Set - 12 Piece Heavy Duty Gardening Tools with Storage Organizer** (Document 5): This tool set has an average rating of 4.6 out of 5 stars and has received over 6,600 reviews. It includes a range of heavy-duty tools and a storage organizer, making it a great gift for serious gardeners.
3. **Garden Set Birthday Gifts for Mom. Wearable Garden Tool Set with Knee Pad & Garden Tools** (Document 2): This gift set has an average rating of 4.5 out of 5 stars and has received over 65 reviews. It includ

## 8. Final Prompt Choice

Based on the prompt experiments, the final prompt was updated to encourage grounded recommendations while also considering ratings, average rating, and number of ratings.

In [14]:
SYSTEM_PROMPT_FINAL = """
You are a helpful Amazon shopping assistant.

Answer the user's question using ONLY the provided Amazon product review and metadata context.
Do not make up details that are not supported by the context.
If the context is not sufficient to answer confidently, say so.

When answering:
- keep the answer concise and practical
- recommend only products supported by the retrieved context
- mention product titles when helpful
- avoid repeating duplicate or near-duplicate products
- give 2 to 4 recommendations when several good options exist
- consider review text together with rating, average rating, and number of ratings when deciding which products seem strongest
- prefer products with stronger overall support when the retrieved context suggests clear differences
- briefly justify each recommendation using evidence from the retrieved context
- avoid assumptions that are not directly supported by the context
"""

In [15]:
# Test pipeline 
query = "What would be a good gift for someone who loves gardening?"

result = run_semantic_rag(
    query=query,
    retriever=semantic_retriever,
    llm=llm,
    system_prompt=SYSTEM_PROMPT_FINAL
)

print(result["answer"])

Based on the provided context, I recommend the following gift options for someone who loves gardening:

1. **Garden Tools Set, JUMPHIGH 10 Pieces Gardening Tools with Purple Floral Print** ($35.99): This 10-piece gardening tool set is highly rated (4.4/5) and includes a variety of tools for different gardening tasks. It's a great gift for gardening enthusiasts, and the floral print design makes it a stylish addition to any garden.
2. **Scuddles Garden Tools Set - 12 Piece Heavy Duty Gardening Tools with Storage Organizer** ($35.99): This 12-piece gardening tool set is also highly rated (4.6/5) and includes a range of heavy-duty tools, including a shovel, rake, and pruner. The storage organizer makes it easy to carry and store the tools.
3. **Funny Combat Garden Gnome Riding a Flamingo – Cute Home Decor Statue** (Price: N/A): While not a traditional gardening gift, this garden gnome is a unique and humorous addition to any garden or outdoor space. It's highly rated (4.0/5) and can be a 

Prompt variant 1 produced more natural and detailed responses, but often introduced unsupported reasoning. Prompt variant 2 was more grounded but sometimes too restrictive. The final prompt provided the best balance, producing concise, evidence-based recommendations while making better use of rating information.

## 9. Validate Final Script Implementation

After exploration in the notebook, the final semantic RAG pipeline was moved into `src/rag_pipeline.py` and tested here by importing the script directly.

In [16]:
from src.rag_pipeline import (
    get_semantic_rag_components,
    run_semantic_rag,
    SYSTEM_PROMPT_FINAL,
)

vectorstore, semantic_retriever, llm = get_semantic_rag_components(top_k=5)

result = run_semantic_rag(
    query="What would be a good gift for someone who loves gardening?",
    retriever=semantic_retriever,
    llm=llm,
    system_prompt=SYSTEM_PROMPT_FINAL,
)

print(result["answer"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7221.53it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded semantic vector store.
Based on the provided Amazon product reviews and metadata, here are some gift recommendations for someone who loves gardening:

1. **Garden Tools Set by JUMPHIGH**: This 10-piece gardening tool kit is a great gift for gardeners, offering a variety of tasks including digging, weeding, raking, and more. It's durable, ergonomic, and comes with a carrying case. ($35.99)
2. **Garden Tools Set by Scuddles**: This 12-piece heavy-duty gardening tool set is a great gift for men and women, offering a range of tools including a shovel, trowel, rake, and more. It comes with a storage organizer and ergonomic handles. ($35.99)

Both of these options are well-rated (4.4 and 4.6 stars, respectively) and offer a range of features that would make them great gifts for gardeners.

If you want to consider a more unique gift, you could also look at the **Mothers Day Gift Basket by Strong Roots**, which offers a thoughtful and personalized gift for gardening enthusiasts. However

## 10. Hybrid RAG Exploration (To Be Completed)

Next step: extend the semantic RAG pipeline by replacing the semantic retriever with a hybrid retriever that combines:

- BM25 keyword retrieval
- semantic retrieval

Planned steps:
1. Implement `src/hybrid.py`
2. Load the hybrid retriever here
3. Run RAG with the hybrid retriever
4. Compare semantic-only and hybrid outputs on the same queries

In [17]:
from src.hybrid import get_hybrid_retriever
from src.rag_pipeline_hybrid import run_hybrid_rag

hybrid_retriever = get_hybrid_retriever(semantic_retriever, bm25)

hybrid_result = run_hybrid_rag(
    query="What would be a good gift for someone who loves gardening?",
    hybrid_retriever=hybrid_retriever,
    llm=llm
)

print(hybrid_result["answer"])

Based on the provided reviews, here are some gift ideas for someone who loves gardening:

1. **Garden Tools Set** by JUMPHIGH (Document 4) - This 10-piece set includes a variety of tools, such as trowel, rake, and pruning shears, and is suitable for both men and women.
2. **Scuddles Garden Tools Set** (Document 5) - This set includes 12 heavy-duty tools, such as a garden shovel, trowel, and rake, and comes with a storage organizer.
3. **Evergreen Garden Gnome University of Arizona** (Document 13) - This garden gnome is a unique and fun gift for any gardener, especially those who love the University of Arizona.
4. **EasyStart Kitchen Garden Herbs** (Document 14) - This kit allows the recipient to grow 6 popular culinary herbs from scratch, making it a great gift for both beginners and experienced gardeners.
5. **CobraCo Expandable Window Planter** (Document 15) - This planter is perfect for growing herbs or small plants in a kitchen window and is a great gift for anyone who loves cookin

## 11. Notes for Discussion and README

Key observations so far:
- Semantic retrieval performed well on both product-specific and more abstract intent-based queries.
- Prompt design had a strong effect on grounding and usefulness.
- Including rating, average rating, and number of ratings improved recommendation quality by helping the model weigh stronger evidence more effectively.
- The final semantic RAG pipeline now works both in the notebook and in `src/rag_pipeline.py`.